# DataFrame の大量テキストをまとめて分かち書きする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/03_japanese_tokenization/tokenize_pandas.ipynb)

NLP クックブック 03-13 ／ 日本語の形態素解析と文分割

数万件のテキスト列を持つ DataFrame を、進捗を見ながら効率よくトークン化したい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import pandas as pd
from fugashi import Tagger
from tqdm.auto import tqdm

df = pd.DataFrame({"text": ["今日は良い天気ですね。", "自然言語処理を勉強しています。", "明日は雨が降るらしい。"] * 10000})
tagger = Tagger("-Owakati")  # Tagger は 1 回だけ作って使い回す (毎回作ると非常に遅い)

tqdm.pandas()
df["wakati"] = df["text"].progress_apply(lambda s: tagger.parse(s).strip())
df.head(3)

,text,wakati
0,今日は良い天気ですね。,今日 は 良い 天気 です ね 。
1,自然言語処理を勉強しています。,自然 言語 処理 を 勉強 し て い ます 。
2,明日は雨が降るらしい。,明日 は 雨 が 降る らしい 。


In [3]:
# 複数コアで並列化 (プロセスごとに Tagger を作る)
from joblib import Parallel, delayed
import numpy as np

def work(texts):
    t = Tagger("-Owakati")
    return [t.parse(s).strip() for s in texts]

chunks = np.array_split(df["text"].to_numpy(), 8)
results = Parallel(n_jobs=2)(delayed(work)(c) for c in chunks)
df["wakati2"] = [s for chunk in results for s in chunk]
print((df["wakati"] == df["wakati2"]).all())

True


- Colab の無料枠は 2 コアなので並列化の効果は 2 倍弱。MeCab は 1 コアでも毎秒数万文は処理できる。